In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers


In [ ]:
# On Colab: Runtime -> Change runtime type -> T4 GPU
# On a shared machine (e.g. a DGX): pin a GPU via `CUDA_VISIBLE_DEVICES=<id>`
# before launching Jupyter, and enable memory growth so TensorFlow doesn't
# preallocate the whole GPU while other people are using the rest of it.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(gpus)

In [ ]:
timesteps = 1000
embedding_dim = 128

In [ ]:
import os
import numpy as np
import tensorflow as tf

# Cache the (airplane-only) dataset on Google Drive so a Colab session
# reset doesn't force a re-download from the internet.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_PATH = "/content/drive/MyDrive/diffusion_cache/cifar10_airplanes.npz"
except ImportError:
    # Not running on Colab (e.g. local machine) - cache next to the notebook.
    CACHE_PATH = "cifar10_airplanes.npz"

os.makedirs(os.path.dirname(CACHE_PATH) or ".", exist_ok=True)

if os.path.exists(CACHE_PATH):
    print(f"Loading cached airplane-only CIFAR-10 from {CACHE_PATH}")
    cached = np.load(CACHE_PATH)
    x_train, y_train = cached["x_train"], cached["y_train"]
    x_test, y_test = cached["x_test"], cached["y_test"]
else:
    print("No cache found, downloading full CIFAR-10 (one-time cost)...")
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    # Keep only one class: airplane (label 0)
    mask_train = y_train.flatten() == 0
    x_train = x_train[mask_train]
    y_train = y_train[mask_train]

    mask_test = y_test.flatten() == 0
    x_test = x_test[mask_test]
    y_test = y_test[mask_test]

    np.savez(CACHE_PATH, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)
    print(f"Cached airplane-only CIFAR-10 to {CACHE_PATH}")

print(f"Shape of x_train (airplanes only): {x_train.shape}")
print(f"Min pixel value: {x_train.min()}, Max pixel value: {x_train.max()}")

In [ ]:
mask = y_train.flatten() == 0
x_train = x_train[mask]
y_train = y_train[mask]

# Normalize pixel values to [-1, 1]
# The reverse process starts from N(0, 1), so the data has to be
# centred on 0 for the forward process to actually end at pure noise.
# Guarded so re-running this cell (e.g. via "Run all" after it already
# ran once) doesn't normalize an already-normalized array.
if x_train.dtype != np.float32 or x_train.max() > 1.5:
    x_train = x_train.astype("float32") / 255.0
    x_train = x_train * 2.0 - 1.0


In [ ]:
assert x_train.min() < -0.9 and x_train.max() > 0.9, \
    f"data range looks wrong: [{x_train.min():.3f}, {x_train.max():.3f}]"

In [ ]:
print(x_train.shape)
print(y_train.shape)
# Display an image


In [ ]:
def to_display(x):
    """Map an image from [-1, 1] back to [0, 1] for plotting."""
    return np.clip((x + 1.0) / 2.0, 0.0, 1.0)


plt.imshow(to_display(x_train[100]))
plt.axis("off")
plt.show()

In [ ]:
def forward_diffusion(x_0, t, beta_schedule):
    alpha = 1 - beta_schedule
    alpha_bar = np.cumprod(alpha)
    noise = np.random.normal(loc = 0, scale = 1,size=x_0.shape)
    x_t = np.sqrt(alpha_bar[t]) * x_0 +np.sqrt(1 - alpha_bar[t]) * noise
    return x_t, noise

In [ ]:
def linear_schedule(timesteps, beta_start = 0.0001, beta_end = 0.02):
    return np.linspace(beta_start, beta_end, timesteps)

In [ ]:
def cosine_schedule(timesteps, s=0.008):
    """Nichol & Dhariwal cosine schedule.

    The linear schedule destroys most of the signal in the first ~200 steps
    at 32x32, so a lot of the model's capacity is spent on timesteps that
    carry no information. The cosine schedule spreads the damage evenly.
    """
    steps = np.arange(timesteps + 1)
    f = np.cos(((steps / timesteps) + s) / (1 + s) * (np.pi / 2)) ** 2
    alpha_bar = f / f[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return np.clip(betas, 0.0001, 0.999)

In [ ]:
def visualise_timesteps(x_0, timesteps, beta_schedule):
    t = np.random.randint(0, timesteps, size = 5)
    t = np.append(t, [0, 5, 10, 50, 100])
    t.sort()
    fig, axes = plt.subplots(1, len(t), figsize=(15, 5))
    for i, timestep in enumerate(t):
        x_t, _ = forward_diffusion(x_0, timestep, beta_schedule)
        axes[i].imshow(to_display(x_t))
        axes[i].set_title(f'Timestep: {timestep}')
        axes[i].axis('off')
    plt.show()

In [ ]:
beta_schedule = cosine_schedule(timesteps)
x_0 = x_train[100]
visualise_timesteps(x_0, timesteps, beta_schedule)

# read these one code block again

In [ ]:
def timestep_positional_encoding(timesteps, embedding_dim):
    pe = np.zeros((timesteps, embedding_dim))
    position = np.arange(0, timesteps, 1)
    power = np.arange(0, embedding_dim, 2) / embedding_dim
    pe[:, 0::2] = np.sin(position[:, np.newaxis] / (10000 ** power))
    pe[:, 1::2] = np.cos(position[:, np.newaxis] / (10000 ** power))
    return pe

In [ ]:
pe = timestep_positional_encoding(timesteps, embedding_dim)
print(pe.shape)

In [ ]:
print(pe[4])

In [ ]:
class ResBlock(layers.Layer):

    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = layers.Conv2D(
            out_channels, 3, padding="same"
        )
        # GroupNorm, not BatchNorm: every image in the batch carries a
        # different noise level, so batch statistics are meaningless here.
        self.norm1 = layers.GroupNormalization(groups=8)

        self.conv2 = layers.Conv2D(
            out_channels, 3, padding="same"
        )
        self.norm2 = layers.GroupNormalization(groups=8)

        self.time_proj = layers.Dense(out_channels)

        self.activation = layers.Activation("swish")

        if in_channels != out_channels:
            self.skip = layers.Conv2D(
                out_channels, 1, padding="same"
            )
        else:
            self.skip = None

    def call(self, x, t_emb):

        if self.skip is not None:
            residual = self.skip(x)
        else:
            residual = x

        h = self.conv1(x)
        h = self.norm1(h)

        time = self.time_proj(t_emb)
        time = time[:, tf.newaxis, tf.newaxis, :]

        h = h + time
        h = self.activation(h)

        h = self.conv2(h)
        h = self.norm2(h)

        h = h + residual
        h = self.activation(h)

        return h

In [ ]:
class AttentionBlock(layers.Layer):
    """Self-attention over the 8x8 feature map.

    Convolutions only see a local window; one attention layer at the
    bottleneck lets every pixel look at every other pixel, which is what
    keeps a generated object globally coherent instead of locally plausible.
    """

    def __init__(self, channels, num_heads=4):
        super().__init__()

        self.norm = layers.GroupNormalization(groups=8)

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=channels // num_heads
        )

    def call(self, x):

        b = tf.shape(x)[0]
        h, w, c = x.shape[1], x.shape[2], x.shape[3]

        y = self.norm(x)
        y = tf.reshape(y, (b, h * w, c))
        y = self.attention(y, y)
        y = tf.reshape(y, (b, h, w, c))

        return x + y

In [ ]:
class DownBlock(layers.Layer):

    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = layers.Conv2D(
            out_channels,
            3,
            strides=2,
            padding="same"
        )

    def call(self, x, t_emb):

        x = self.resblock(x, t_emb)

        skip = x

        x = self.downsample(x)

        return x, skip

In [ ]:
class UpBlock(layers.Layer):

    def __init__(self, in_channels, skip_channels, out_channels, time_dim):
        super().__init__()

        self.upsample = layers.Conv2DTranspose(
            out_channels,
            3,
            strides=2,
            padding="same"
        )

        self.resblock = ResBlock(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def call(self, x, skip, t_emb):

        x = self.upsample(x)

        x = tf.concat([x, skip], axis=-1)

        x = self.resblock(x, t_emb)

        return x

In [ ]:
class UNet(tf.keras.Model):

    def __init__(self, input_shape, time_dim=128, base=32):
        super().__init__()

        self.time_dim = time_dim

        c1, c2, c3 = base, base * 2, base * 4

        # Let the network reshape the sinusoidal encoding itself
        self.time_mlp = tf.keras.Sequential([
            layers.Dense(time_dim * 2, activation="swish"),
            layers.Dense(time_dim * 2),
        ])

        # Input
        self.input_conv = layers.Conv2D(
            c1,
            3,
            padding="same"
        )

        # Downsampling
        self.down1 = DownBlock(
            c1, c2, time_dim
        )

        self.down2 = DownBlock(
            c2, c3, time_dim
        )

        # Bottleneck
        self.bottleneck1 = ResBlock(
            c3, c3, time_dim
        )

        self.attention = AttentionBlock(c3)

        self.bottleneck2 = ResBlock(
            c3, c3, time_dim
        )

        # Upsampling
        self.up2 = UpBlock(
            c3, c3, c2, time_dim
        )

        self.up1 = UpBlock(
            c2, c2, c1, time_dim
        )

        # Output
        self.output_conv = layers.Conv2D(
            3,
            3,
            padding="same"
        )

    def call(self, inputs):

        x, t_emb = inputs

        t_emb = self.time_mlp(t_emb)

        # Input
        x = self.input_conv(x)

        # Down
        x, skip1 = self.down1(x, t_emb)
        x, skip2 = self.down2(x, t_emb)

        # Bottleneck
        x = self.bottleneck1(x, t_emb)
        x = self.attention(x)
        x = self.bottleneck2(x, t_emb)

        # Up
        x = self.up2(x, skip2, t_emb)
        x = self.up1(x, skip1, t_emb)

        # Output
        x = self.output_conv(x)

        return x

## Next 2 blocks are just for testing no relevance

In [ ]:
batch_size = 4

x_dummy = tf.random.normal(
    (batch_size, 32, 32, 3)
)

t_test = np.random.randint(
    0, timesteps, size=batch_size
)

t_emb_test = timestep_positional_encoding(
    timesteps,
    embedding_dim
)[t_test]

t_emb_test = tf.convert_to_tensor(
    t_emb_test,
    dtype=tf.float32
)

In [ ]:
model = UNet(
    input_shape=(32, 32, 3),
    time_dim=embedding_dim,
    base=32
)

predicted_noise = model(
    [x_dummy, t_emb_test]
)

print("Input:", x_dummy.shape)
print("Time embedding:", t_emb_test.shape)
print("Output:", predicted_noise.shape)
print("Parameters:", f"{model.count_params():,}")

In [ ]:
mse = tf.keras.losses.MeanSquaredError()

def loss_fn(y_true, y_pred):
    return mse(y_true, y_pred)

In [ ]:
def get_noisy_image(x_0, beta_schedule):

    batch_size = x_0.shape[0]

    x_0 = tf.cast(x_0, tf.float32)
    beta_schedule = tf.cast(beta_schedule, tf.float32)

    t = tf.random.uniform(
        shape=(batch_size,),
        minval=0,
        maxval=len(beta_schedule),
        dtype=tf.int32
    )

    alpha = 1 - beta_schedule
    alpha_bar = tf.math.cumprod(alpha)

    noise = tf.random.normal(
        shape=tf.shape(x_0),
        mean=0.0,
        stddev=1.0
    )

    alpha_bar_t = tf.gather(alpha_bar, t)
    alpha_bar_t = alpha_bar_t[:, tf.newaxis, tf.newaxis, tf.newaxis]

    x_t = (
        tf.sqrt(alpha_bar_t) * x_0
        + tf.sqrt(1 - alpha_bar_t) * noise
    )

    return x_t, noise, t

In [ ]:
def get_gradients(model, x_0, beta_schedule):

    x_t, noise, t = get_noisy_image(
        x_0,
        beta_schedule
    )

    t_emb = timestep_positional_encoding(
        len(beta_schedule),
        model.time_dim
    )[t]

    t_emb = tf.convert_to_tensor(
        t_emb,
        dtype=tf.float32
    )

    with tf.GradientTape() as tape:

        predicted_noise = model(
            [x_t, t_emb],
            training=True
        )

        loss = loss_fn(
            noise,
            predicted_noise
        )

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    return loss, gradients

In [ ]:
loss, gradients = get_gradients(
    model,
    x_train[:4],
    beta_schedule
)

print("Loss:", loss.numpy())

for i, grad in enumerate(gradients):
    print(i, grad.shape)

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-4
)

# Create Adam's moment variables now, outside tf.function. Newer Keras
# builds them lazily on the first apply_gradients, which can fail inside
# a tf.function ('Adam' object has no attribute '_momentums'). Building
# here also lets the checkpoint below save/restore them.
optimizer.build(model.trainable_variables)

# Exponential moving average of the weights.
# Sampling from the EMA copy instead of the live weights is the single
# biggest visual-quality win in diffusion training.
# Re-run this cell if you ever rebuild `model`.
ema_decay = 0.999

ema_weights = [
    tf.Variable(v, trainable=False)
    for v in model.trainable_variables
]

ema_step = tf.Variable(0.0, trainable=False)


def swap_in_ema():
    """Move EMA weights into the model, returning the live weights."""
    backup = [tf.identity(v) for v in model.trainable_variables]
    for v, e in zip(model.trainable_variables, ema_weights):
        v.assign(e)
    return backup


def restore_weights(backup):
    for v, b in zip(model.trainable_variables, backup):
        v.assign(b)

In [ ]:
# Checkpoint to Drive periodically so a dropped/expired Colab session
# (or a git pull that reruns cells) only costs a few epochs, not the
# whole run. Re-running this cell auto-resumes from the latest checkpoint.
CKPT_DIR = os.path.join(os.path.dirname(CACHE_PATH), "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

checkpoint = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer,
    ema_weights=ema_weights,
    ema_step=ema_step,
    epoch=tf.Variable(0, dtype=tf.int64, trainable=False),
)
ckpt_manager = tf.train.CheckpointManager(checkpoint, CKPT_DIR, max_to_keep=3)

start_epoch = 0
history = []
history_path = os.path.join(CKPT_DIR, "history.npy")

if ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint).expect_partial()
    start_epoch = int(checkpoint.epoch.numpy())
    if os.path.exists(history_path):
        history = list(np.load(history_path))
    print(f"Resumed from checkpoint: epoch {start_epoch}, {len(history)} loss values loaded")
else:
    print("No checkpoint found, starting fresh.")


def save_checkpoint(epoch):
    checkpoint.epoch.assign(epoch)
    ckpt_manager.save()
    np.save(history_path, np.array(history))

In [ ]:
PE = tf.constant(
    timestep_positional_encoding(timesteps, embedding_dim),
    dtype=tf.float32
)

BETA = tf.constant(beta_schedule, dtype=tf.float32)
ALPHA = 1.0 - BETA
ALPHA_BAR = tf.math.cumprod(ALPHA)
ALPHA_BAR_PREV = tf.concat([[1.0], ALPHA_BAR[:-1]], axis=0)


@tf.function(autograph=False)
def train_step(x_0):

    batch = tf.shape(x_0)[0]

    t = tf.random.uniform(
        (batch,), 0, timesteps, dtype=tf.int32
    )

    noise = tf.random.normal(tf.shape(x_0))

    alpha_bar_t = tf.gather(ALPHA_BAR, t)[:, None, None, None]

    x_t = (
        tf.sqrt(alpha_bar_t) * x_0
        + tf.sqrt(1.0 - alpha_bar_t) * noise
    )

    t_emb = tf.gather(PE, t)

    with tf.GradientTape() as tape:

        predicted_noise = model([x_t, t_emb], training=True)

        loss = loss_fn(noise, predicted_noise)

    gradients = tape.gradient(loss, model.trainable_variables)

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    # EMA update, with a warm-up so the average is usable early on
    ema_step.assign_add(1.0)
    d = tf.minimum(ema_decay, (1.0 + ema_step) / (10.0 + ema_step))
    for e, v in zip(ema_weights, model.trainable_variables):
        e.assign(d * e + (1.0 - d) * v)

    return loss

## Sampling

Defined before training so the training loop can preview samples as it goes.

In [ ]:
def show_images(images, title=None, cols=8):
    n = len(images)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
    flat = np.array(axes).reshape(-1)
    for ax, img in zip(flat, images):
        ax.imshow(img)
        ax.axis("off")
    for ax in flat[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_trajectory(trajectory):
    n_images = trajectory[0][1].shape[0]
    n_steps = len(trajectory)
    fig, axes = plt.subplots(n_images, n_steps,
                             figsize=(n_steps * 1.4, n_images * 1.4))
    axes = np.array(axes).reshape(n_images, n_steps)
    for col, (t, imgs) in enumerate(trajectory):
        for row in range(n_images):
            axes[row, col].imshow(imgs[row])
            axes[row, col].axis("off")
        axes[0, col].set_title(f"t={t}", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
@tf.function
def predict(x, t_emb):
    return model([x, t_emb], training=False)


def ddim_sample(n_images=8, image_shape=(32, 32, 3), n_steps=100, eta=0.0):

    step_indices = np.linspace(0, timesteps - 1, n_steps).round().astype(int)[::-1]

    x_t = tf.random.normal((n_images,) + image_shape)

    for i, t in enumerate(step_indices):

        t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else -1

        alpha_bar_t = ALPHA_BAR[t]
        alpha_bar_prev = ALPHA_BAR[t_prev] if t_prev >= 0 else tf.constant(1.0)

        t_emb = tf.tile(PE[t][tf.newaxis, :], [n_images, 1])

        predicted_noise = predict(x_t, t_emb)

        x_0_pred = (
            x_t - tf.sqrt(1.0 - alpha_bar_t) * predicted_noise
        ) / tf.sqrt(alpha_bar_t)
        x_0_pred = tf.clip_by_value(x_0_pred, -1.0, 1.0)

        sigma = eta * tf.sqrt(
            (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)
            * (1.0 - alpha_bar_t / alpha_bar_prev)
        )

        direction = tf.sqrt(
            tf.maximum(1.0 - alpha_bar_prev - sigma ** 2, 0.0)
        ) * predicted_noise

        x_t = tf.sqrt(alpha_bar_prev) * x_0_pred + direction

        if eta > 0 and t_prev >= 0:
            x_t += sigma * tf.random.normal(tf.shape(x_t))

    return to_display(x_t.numpy())

In [ ]:
def ddpm_sample(model, n_images=8, image_shape=(32, 32, 3),
                return_trajectory=False, snapshot_every=100):

    x_t = tf.random.normal((n_images,) + image_shape)

    trajectory = []

    for t in reversed(range(timesteps)):

        t_emb = tf.tile(PE[t][tf.newaxis, :], [n_images, 1])

        predicted_noise = predict(x_t, t_emb)

        # Clip the predicted x0 every step (matches ddim_sample) so a bad
        # noise prediction can't compound unbounded over 1000 sequential
        # steps.
        x0_pred = (
            x_t - tf.sqrt(1.0 - ALPHA_BAR[t]) * predicted_noise
        ) / tf.sqrt(ALPHA_BAR[t])
        x0_pred = tf.clip_by_value(x0_pred, -1.0, 1.0)

        mean = (
            tf.sqrt(ALPHA_BAR_PREV[t]) * BETA[t] / (1.0 - ALPHA_BAR[t]) * x0_pred
            + tf.sqrt(ALPHA[t]) * (1.0 - ALPHA_BAR_PREV[t]) / (1.0 - ALPHA_BAR[t]) * x_t
        )

        if t > 0:
            variance = (
                BETA[t] * (1.0 - ALPHA_BAR_PREV[t]) / (1.0 - ALPHA_BAR[t])
            )
            x_t = mean + tf.sqrt(variance) * tf.random.normal(tf.shape(x_t))
        else:
            x_t = mean

        if return_trajectory and (t % snapshot_every == 0 or t == timesteps - 1):
            trajectory.append((t, to_display(x_t.numpy())))

    images = to_display(x_t.numpy())

    if return_trajectory:
        return images, trajectory

    return images

In [ ]:
assert x_train.min() < -0.9 and x_train.max() > 0.9, \
    f"data range looks wrong: [{x_train.min():.3f}, {x_train.max():.3f}]"

## Train

On a Colab T4 expect roughly 5-10 s/epoch, so 400 epochs is about 30-60 minutes.
Loss should fall to around 0.03-0.04; below ~0.05 the samples start showing sky and fuselage shapes.
The preview every 100 epochs uses the EMA weights, so you can stop early if it already looks good.

In [ ]:
epochs = 100
batch_size = 128
checkpoint_every = 5  # epochs; keep this low so a dropped session loses little

dataset = (
    tf.data.Dataset
    .from_tensor_slices(x_train)
    .map(
        lambda im: tf.image.random_flip_left_right(im),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .shuffle(5000)
    .batch(batch_size, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

for epoch in range(start_epoch, epochs):

    losses = [float(train_step(x_batch)) for x_batch in dataset]
    history.append(np.mean(losses))

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}, Loss: {history[-1]:.4f}")

    if (epoch + 1) % checkpoint_every == 0 or epoch == epochs - 1:
        save_checkpoint(epoch + 1)

    # Peek at samples while it trains
    if (epoch + 1) % 100 == 0:
        backup = swap_in_ema()
        show_images(
            ddim_sample(8, n_steps=100),
            title=f"epoch {epoch + 1}",
            cols=8
        )
        restore_weights(backup)

## Generate

In [ ]:
backup = swap_in_ema()

samples = ddpm_sample(model, n_images=16)

show_images(samples, title="Generated airplanes (DDPM, 1000 steps, EMA)")

restore_weights(backup)

In [ ]:
backup = swap_in_ema()

_, trajectory = ddpm_sample(
    model,
    n_images=3,
    return_trajectory=True,
    snapshot_every=100
)

show_trajectory(trajectory)

restore_weights(backup)

In [ ]:
backup = swap_in_ema()

samples_fast = ddim_sample(16, n_steps=100)

show_images(samples_fast, title="Generated airplanes (DDIM, 100 steps, EMA)")

restore_weights(backup)

In [ ]:
# Save both the live weights and the EMA copy
model.save_weights("diffusion_airplanes.weights.h5")

backup = swap_in_ema()
model.save_weights("diffusion_airplanes_ema.weights.h5")
restore_weights(backup)

# To reload later:
# model = UNet(input_shape=(32, 32, 3), time_dim=embedding_dim, base=32)
# model([tf.zeros((1, 32, 32, 3)), tf.zeros((1, embedding_dim))])   # build first
# model.load_weights("diffusion_airplanes_ema.weights.h5")

## Blur + Noise Diffusion (Cold Diffusion style)

A second, self-contained pipeline: the forward process blurs *and* adds
noise together at each timestep (instead of noise alone), and one model is
trained to reverse both jointly. It shares the dataset and the `ALPHA_BAR`
schedule from above, but has its own model, optimizer, checkpoint, training
loop, and sampler — nothing above this point is modified, so the Gaussian
pipeline stays intact for comparison.

Since the degradation isn't purely additive noise, the model is trained to
predict `x0` directly (Cold Diffusion's approach for compound degradations)
rather than predicting noise.

In [ ]:
SIGMA_MAX = 2.5
KERNEL_SIZE = 17


def gaussian_kernel_2d(sigma, size):
    ax = np.arange(-(size // 2), size // 2 + 1, dtype=np.float32)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2.0 * max(sigma, 1e-3) ** 2))
    kernel /= kernel.sum()
    return kernel


# Blur sigma ramps up on the same schedule as the noise (alpha_bar), so both
# forms of corruption reach full strength together at the last timestep.
blur_sigma = SIGMA_MAX * (1.0 - np.sqrt(ALPHA_BAR.numpy()))

BLUR_KERNELS = tf.constant(
    np.stack([gaussian_kernel_2d(s, KERNEL_SIZE) for s in blur_sigma]),
    dtype=tf.float32
)  # shape: (timesteps, KERNEL_SIZE, KERNEL_SIZE)

print("BLUR_KERNELS shape:", BLUR_KERNELS.shape)

In [ ]:
def apply_blur(x, t_indices):
    """Blur every image in the batch by its own per-example kernel (indexed
    by t), in a single depthwise convolution.

    The trick: fold the batch dimension into the channel dimension so each
    example gets its own filter, since tf.nn.depthwise_conv2d applies one
    filter per input channel.
    """
    b = tf.shape(x)[0]
    h, w, c = x.shape[1], x.shape[2], x.shape[3]

    kernels = tf.gather(BLUR_KERNELS, t_indices)                    # [B, K, K]
    kernels = tf.repeat(kernels[:, :, :, tf.newaxis], c, axis=-1)   # [B, K, K, C]
    kernels = tf.transpose(kernels, [1, 2, 0, 3])                   # [K, K, B, C]
    kernels = tf.reshape(kernels, [KERNEL_SIZE, KERNEL_SIZE, b * c, 1])

    x_r = tf.transpose(x, [1, 2, 0, 3])                             # [H, W, B, C]
    x_r = tf.reshape(x_r, [1, h, w, b * c])

    blurred = tf.nn.depthwise_conv2d(
        x_r, kernels, strides=[1, 1, 1, 1], padding="SAME"
    )                                                                # [1, H, W, B*C]

    blurred = tf.reshape(blurred, [h, w, b, c])
    blurred = tf.transpose(blurred, [2, 0, 1, 3])                   # [B, H, W, C]
    return blurred


def degrade_blur_noise(x0, t_indices, noise):
    """Combined corruption: blur x0 by the timestep's sigma, then blend with
    noise using the same alpha_bar weighting as the Gaussian pipeline."""
    alpha_bar_t = tf.gather(ALPHA_BAR, t_indices)[:, tf.newaxis, tf.newaxis, tf.newaxis]
    blurred = apply_blur(x0, t_indices)
    return tf.sqrt(alpha_bar_t) * blurred + tf.sqrt(1.0 - alpha_bar_t) * noise

In [ ]:
model_blur = UNet(
    input_shape=(32, 32, 3),
    time_dim=embedding_dim,
    base=32
)

# Build the model so its weights exist before EMA/checkpoint setup.
model_blur([tf.zeros((1, 32, 32, 3)), tf.zeros((1, embedding_dim))])

optimizer_blur = tf.keras.optimizers.Adam(learning_rate=2e-4)
optimizer_blur.build(model_blur.trainable_variables)   # see the Gaussian optimizer cell

ema_weights_blur = [
    tf.Variable(v, trainable=False)
    for v in model_blur.trainable_variables
]

ema_step_blur = tf.Variable(0.0, trainable=False)


def swap_in_ema_blur():
    backup = [tf.identity(v) for v in model_blur.trainable_variables]
    for v, e in zip(model_blur.trainable_variables, ema_weights_blur):
        v.assign(e)
    return backup


def restore_weights_blur(backup):
    for v, b in zip(model_blur.trainable_variables, backup):
        v.assign(b)

In [ ]:
# Checkpointing for the blur model, mirroring the Gaussian pipeline's setup
# (same Drive-backed auto-resume behavior).
CKPT_DIR_BLUR = os.path.join(os.path.dirname(CACHE_PATH), "checkpoints_blur")
os.makedirs(CKPT_DIR_BLUR, exist_ok=True)

checkpoint_blur = tf.train.Checkpoint(
    model=model_blur,
    optimizer=optimizer_blur,
    ema_weights=ema_weights_blur,
    ema_step=ema_step_blur,
    epoch=tf.Variable(0, dtype=tf.int64, trainable=False),
)
ckpt_manager_blur = tf.train.CheckpointManager(checkpoint_blur, CKPT_DIR_BLUR, max_to_keep=3)

start_epoch_blur = 0
history_blur = []
history_blur_path = os.path.join(CKPT_DIR_BLUR, "history.npy")

if ckpt_manager_blur.latest_checkpoint:
    checkpoint_blur.restore(ckpt_manager_blur.latest_checkpoint).expect_partial()
    start_epoch_blur = int(checkpoint_blur.epoch.numpy())
    if os.path.exists(history_blur_path):
        history_blur = list(np.load(history_blur_path))
    print(f"Resumed blur model from checkpoint: epoch {start_epoch_blur}, {len(history_blur)} loss values loaded")
else:
    print("No blur model checkpoint found, starting fresh.")


def save_checkpoint_blur(epoch):
    checkpoint_blur.epoch.assign(epoch)
    ckpt_manager_blur.save()
    np.save(history_blur_path, np.array(history_blur))

In [ ]:
@tf.function(autograph=False)
def train_step_blur(x_0):

    batch = tf.shape(x_0)[0]

    t = tf.random.uniform(
        (batch,), 0, timesteps, dtype=tf.int32
    )

    noise = tf.random.normal(tf.shape(x_0))

    x_t = degrade_blur_noise(x_0, t, noise)

    t_emb = tf.gather(PE, t)

    with tf.GradientTape() as tape:

        x0_pred = model_blur([x_t, t_emb], training=True)

        # x0-prediction objective: the combined blur+noise degradation
        # isn't purely additive, so predicting "the noise" wouldn't invert
        # it the way it does in the Gaussian pipeline.
        loss = loss_fn(x_0, x0_pred)

    gradients = tape.gradient(loss, model_blur.trainable_variables)

    optimizer_blur.apply_gradients(
        zip(gradients, model_blur.trainable_variables)
    )

    ema_step_blur.assign_add(1.0)
    d = tf.minimum(ema_decay, (1.0 + ema_step_blur) / (10.0 + ema_step_blur))
    for e, v in zip(ema_weights_blur, model_blur.trainable_variables):
        e.assign(d * e + (1.0 - d) * v)

    return loss

In [ ]:
@tf.function
def predict_blur(x, t_emb):
    return model_blur([x, t_emb], training=False)


def cold_sample(n_images=8, image_shape=(32, 32, 3), n_steps=None):
    """Cold-Diffusion-style restart sampler for the combined blur+noise
    degradation. At every step it re-estimates (and clips) x0, then moves
    from x_t to x_{t-1} using the difference between the degradation
    applied at t and at t-1 - the same trick that keeps ddim_sample stable,
    generalized to a degradation that isn't pure noise. x_t is initialized
    to the SAME noise realization that degrade() reuses at every step, so
    the x_t - degrade(x0_hat, t) term cancels almost completely at the
    first step (t=999, where alpha_bar~0 and the degradation is ~pure
    noise) instead of injecting an unrelated Gaussian draw.
    """
    if n_steps is None:
        n_steps = timesteps

    step_indices = np.linspace(0, timesteps - 1, n_steps).round().astype(int)[::-1]

    fixed_noise = tf.random.normal((n_images,) + image_shape)
    x_t = fixed_noise

    for i, t in enumerate(step_indices):

        t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else -1

        t_batch = tf.fill([n_images], t)
        t_emb = tf.tile(PE[t][tf.newaxis, :], [n_images, 1])

        x0_hat = predict_blur(x_t, t_emb)
        x0_hat = tf.clip_by_value(x0_hat, -1.0, 1.0)

        if t_prev >= 0:
            d_t = degrade_blur_noise(x0_hat, t_batch, fixed_noise)
            t_prev_batch = tf.fill([n_images], t_prev)
            d_prev = degrade_blur_noise(x0_hat, t_prev_batch, fixed_noise)
            x_t = x_t - d_t + d_prev
        else:
            x_t = x0_hat

    return to_display(x_t.numpy())

### Train the blur+noise model

Separate model from the Gaussian one above, so this needs its own full
training run - no shortcut from the existing weights. Uses the same
`dataset` defined earlier.

In [ ]:
epochs_blur = 100
checkpoint_every_blur = 5

for epoch in range(start_epoch_blur, epochs_blur):

    losses = [float(train_step_blur(x_batch)) for x_batch in dataset]
    history_blur.append(np.mean(losses))

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}, Loss: {history_blur[-1]:.4f}")

    if (epoch + 1) % checkpoint_every_blur == 0 or epoch == epochs_blur - 1:
        save_checkpoint_blur(epoch + 1)

    if (epoch + 1) % 100 == 0:
        backup = swap_in_ema_blur()
        show_images(
            cold_sample(8, n_steps=100),
            title=f"blur+noise, epoch {epoch + 1}",
            cols=8
        )
        restore_weights_blur(backup)

### Compare: Gaussian DDIM vs blur+noise Cold Diffusion

In [ ]:
backup = swap_in_ema()
gaussian_samples = ddim_sample(8, n_steps=100)
restore_weights(backup)

backup_blur = swap_in_ema_blur()
blur_samples = cold_sample(8, n_steps=100)
restore_weights_blur(backup_blur)

show_images(gaussian_samples, title="Gaussian DDIM (100 steps, EMA)", cols=8)
show_images(blur_samples, title="Blur + Noise Cold Diffusion (100 steps, EMA)", cols=8)

## Paper-style DDPM (Ho et al. 2020) on full CIFAR-10: linear vs cosine schedule

This section reproduces the paper's CIFAR-10 setup and compares the two noise schedules.
The model, training loop, samplers and metrics live in `ddpm_paper.py`; training runs
as SLURM jobs (never directly on the node), and the cells below read the results.

- **Model:** U-Net with 35.7M parameters (base width 128, multipliers 1-2-2-2, 2 residual blocks per level, attention at 16x16, dropout 0.1).
- **Training:** all 50,000 CIFAR-10 images, batch 128, Adam 2e-4, EMA 0.9999, random flips, noise-prediction MSE loss, 1000 diffusion steps. Speed additions that are not in the paper text: bf16 + XLA, 5000-step learning-rate warm-up, gradient-norm clipping at 1.0.
- **Schedules:** `linear` (paper: beta 1e-4 to 0.02) and `cosine` (Nichol & Dhariwal), one model each, saved under `runs/linear` and `runs/cosine`.
- **Metrics:** FID and Inception Score, computed with Keras' InceptionV3 (ImageNet weights). Absolute values therefore differ a little from published ones; compare against the real-data reference numbers in `fid_reference.json`.

Run it (one job per schedule, 50k steps first, then re-submit with a larger step count to continue the same run):

```
sbatch --job-name=ddpm_linear paper_train.sbatch linear 50000 --eval-every 10000 --final-samples 10000 --final-sampler ddpm
sbatch --job-name=ddpm_cosine paper_train.sbatch cosine 50000 --eval-every 10000 --final-samples 10000 --final-sampler ddpm
```

In [ ]:
# ddpm_paper.py only exists in the repo; on a plain Colab upload this section is skipped
try:
    import ddpm_paper as dp
except ImportError:
    dp = None
    print("ddpm_paper.py not found, skipping the schedule plots")

if dp is not None:

    schedules = {name: dp.make_schedule(name) for name in ("linear", "cosine")}

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for name, s in schedules.items():
        ab = s["alpha_bar"]
        ax[0].plot(ab, label=name)
        ax[1].plot(np.log10(ab / (1.0 - ab)), label=name)
    ax[0].set_title("alpha_bar(t): how much of the image survives")
    ax[1].set_title("log10 signal-to-noise ratio")
    for a in ax:
        a.set_xlabel("t")
        a.legend()
    plt.tight_layout()
    plt.show()

    # The same image and the same noise, corrupted by each schedule
    ts = [0, 100, 250, 500, 750, 999]
    x0 = x_train[100]
    noise = np.random.normal(size=x0.shape)
    fig, axes = plt.subplots(2, len(ts), figsize=(2 * len(ts), 4.4))
    for r, (name, s) in enumerate(schedules.items()):
        for c, t in enumerate(ts):
            ab = s["alpha_bar"][t]
            axes[r, c].imshow(to_display(np.sqrt(ab) * x0 + np.sqrt(1 - ab) * noise))
            axes[r, c].axis("off")
            axes[r, c].set_title(f"{name}, t={t}", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
import json
import os


def load_run(name):
    d = os.path.join("runs", name)
    mp, lp = os.path.join(d, "metrics.json"), os.path.join(d, "loss.csv")
    metrics = json.load(open(mp)) if os.path.exists(mp) else []
    loss = np.loadtxt(lp, delimiter=",", skiprows=1, ndmin=2) if os.path.exists(lp) else None
    return metrics, loss


runs = {name: load_run(name) for name in ("linear", "cosine")}

if os.path.exists("fid_reference.json"):
    ref = json.load(open("fid_reference.json"))
    print("Real-data reference (same Inception network):")
    print(f"  FID of the CIFAR-10 test set vs the training set: {ref['fid_test_vs_train']:.2f}")
    print(f"  Inception Score of the training set: {ref['is_train'][0]:.2f}")
    print()

if all(m == [] and l is None for m, l in runs.values()):
    print("No results yet. Submit the two jobs from the markdown cell above.")
else:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for name, (metrics, loss) in runs.items():
        if loss is not None:
            k = min(10, len(loss))
            smooth = np.convolve(loss[:, 1], np.ones(k) / k, mode="valid")
            ax[0].plot(loss[k - 1:, 0], smooth, label=name)
        mon = [m for m in metrics if m["tag"] == "monitor"]
        if mon:
            ax[1].plot([m["step"] for m in mon], [m["fid"] for m in mon], "o-", label=name)
            ax[2].plot([m["step"] for m in mon], [m["is_mean"] for m in mon], "o-", label=name)
    ax[0].set_title("training loss (noise MSE)")
    ax[1].set_title("FID during training (DDIM 100 steps, 5k samples)")
    ax[2].set_title("Inception Score during training")
    for a in ax:
        a.set_xlabel("training step")
        a.legend()
    plt.tight_layout()
    plt.show()

    print(f"{'schedule':9s} {'step':>7s} {'sampler':>8s} {'samples':>8s} {'FID':>8s} {'IS':>13s}")
    for name, (metrics, _) in runs.items():
        for m in [m for m in metrics if m["tag"] in ("final", "eval")]:
            print(f"{name:9s} {m['step']:7d} {m['sampler']:>8s} {m['n_samples']:8d} "
                  f"{m['fid']:8.2f} {m['is_mean']:7.2f} +- {m['is_std']:.2f}")

In [ ]:
import glob

fig, axes = plt.subplots(1, 2, figsize=(14, 7.3))
for a, name in zip(axes, ("linear", "cosine")):
    files = sorted(glob.glob(os.path.join("runs", name, "samples_*step*.png")),
                   key=lambda p: int(p.rsplit("step", 1)[1].split(".")[0]))
    if files:
        a.imshow(plt.imread(files[-1]))
        a.set_title(f"{name}: {os.path.basename(files[-1])}")
    else:
        a.set_title(f"{name}: no final samples yet")
    a.axis("off")
plt.tight_layout()
plt.show()

# This is new part

**Path U-Nets.** Instead of one fixed U shape, the network is described by the list of levels it walks through
(level 0 = 32x32, 1 = 16x16, 2 = 8x8, 3 = 4x4). Every time it lands on a level it runs a ResBlock there;
if it has been on that level before, the last feature map from that visit is concatenated in first
(that is the skip connection). For `[0, 1, 2, 3, 2, 1, 0]` the rule gives an ordinary U-Net.

The idea being tested: go down 1 and back up, then down 2 and back up, ..., then down to the bottleneck and back up,
either one level at a time (`ladder_step`) or in a single jump (`ladder_jump`), plus a few other shapes.

**Comparison.** Every variant trains for the same number of steps as the Gaussian `UNet` above
(same data pipeline, batch 128, Adam 2e-4, EMA). Then the baseline and every variant are scored with
**FID and Inception Score on the same number of samples, the same starting noise and the same DDIM sampler**,
so the numbers can be compared directly instead of judging sample grids by eye.

Everything new is saved next to the existing cache on Drive (the folder of `CACHE_PATH`), so a dropped
Colab session resumes where it stopped. The cells above are unchanged except for the smaller `UNet` (base 32).

In [ ]:
# Building blocks for the path U-Nets. Named PResBlock / PAttentionBlock so the
# ResBlock / AttentionBlock used by UNet above are not overwritten.
import gc
import json
import time
import scipy.linalg

def timestep_embedding(t, dim):
    """Sinusoidal embedding of integer timesteps, shape (B,) -> (B, dim)."""
    t = tf.cast(tf.reshape(t, (-1, 1)), tf.float32)
    half = dim // 2
    freqs = tf.exp(-np.log(10000.0) * tf.range(half, dtype=tf.float32) / half)
    args = t * freqs[tf.newaxis, :]
    return tf.concat([tf.sin(args), tf.cos(args)], axis=-1)


class PResBlock(layers.Layer):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.norm1 = layers.GroupNormalization(groups=8)
        self.conv1 = layers.Conv2D(out_ch, 3, padding="same")
        self.time_proj = layers.Dense(out_ch)
        self.norm2 = layers.GroupNormalization(groups=8)
        self.conv2 = layers.Conv2D(out_ch, 3, padding="same")
        self.skip = layers.Conv2D(out_ch, 1) if in_ch != out_ch else None

    def call(self, x, t_emb):
        h = self.conv1(tf.nn.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, tf.newaxis, tf.newaxis, :]
        h = self.conv2(tf.nn.silu(self.norm2(h)))
        return h + (self.skip(x) if self.skip is not None else x)


class PAttentionBlock(layers.Layer):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = layers.GroupNormalization(groups=8)
        self.attention = layers.MultiHeadAttention(num_heads, channels // num_heads)

    def call(self, x):
        b = tf.shape(x)[0]
        h, w, c = x.shape[1], x.shape[2], x.shape[3]
        y = tf.reshape(self.norm(x), (b, h * w, c))
        y = self.attention(y, y)
        return x + tf.reshape(y, (b, h, w, c))


class Resample(layers.Layer):
    """Move between levels in ONE hop, however many levels apart they are.

    down k levels: a single conv with kernel = stride = 2**k (patchify)
    up   k levels: nearest upsample by 2**k, then a 3x3 conv
    """
    def __init__(self, from_level, to_level, out_ch):
        super().__init__()
        k = to_level - from_level
        if k > 0:
            self.op = layers.Conv2D(out_ch, 2 ** k, strides=2 ** k)
        else:
            self.op = tf.keras.Sequential([
                layers.UpSampling2D(2 ** (-k), interpolation="nearest"),
                layers.Conv2D(out_ch, 3, padding="same"),
            ])

    def call(self, x):
        return self.op(x)

In [ ]:
class PathUNet(tf.keras.Model):
    """A U-Net whose shape is just a list of levels to walk through.

    level 0 = 32x32, 1 = 16x16, 2 = 8x8, 3 = 4x4.
    Every time we land on a level we run a ResBlock there. If we've been on
    that level before, the last feature map from that visit is concatenated
    in first -- that is the skip connection. For [0,1,2,3,2,1,0] this rule
    gives exactly the normal U-Net.

    Takes integer timesteps directly: model([x_t, t]).
    """
    def __init__(self, path, base=32, mults=(1, 2, 2, 4), time_dim=128, attn_levels=(3,)):
        super().__init__()
        assert path[0] == 0 and path[-1] == 0, "path must start and end at full resolution"
        self.levels = list(path)          # not self.path -- Keras reserves that name
        self.time_dim = time_dim
        ch = [base * m for m in mults]

        self.time_mlp = tf.keras.Sequential([
            layers.Dense(time_dim * 4, activation="swish"),
            layers.Dense(time_dim * 4),
        ])
        self.input_conv = layers.Conv2D(ch[0], 3, padding="same")

        # Walk the path once here just to work out channel counts per block
        self.moves, self.blocks, self.attns = [], [], []
        seen = {0: ch[0]}
        cur = 0
        self.blocks.append(PResBlock(ch[0], ch[0]))           # first visit to level 0
        for nxt in self.levels[1:]:
            self.moves.append(Resample(cur, nxt, ch[nxt]))
            in_ch = ch[nxt] + seen.get(nxt, 0)                # + skip if level was visited
            self.blocks.append(PResBlock(in_ch, ch[nxt]))
            self.attns.append(PAttentionBlock(ch[nxt]) if nxt in attn_levels else None)
            seen[nxt] = ch[nxt]
            cur = nxt

        self.out_norm = layers.GroupNormalization(groups=8)
        self.output_conv = layers.Conv2D(3, 3, padding="same")

    def call(self, inputs):
        x, t = inputs
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim))

        x = self.blocks[0](self.input_conv(x), t_emb)
        memory = {0: x}                                       # last feature map seen at each level
        for nxt, move, block, attn in zip(self.levels[1:], self.moves, self.blocks[1:], self.attns):
            x = move(x)
            if nxt in memory:
                x = tf.concat([x, memory[nxt]], axis=-1)
            x = block(x, t_emb)
            if attn is not None:
                x = attn(x)
            memory[nxt] = x

        return self.output_conv(tf.nn.silu(self.out_norm(x)))

In [ ]:
PATH_BASE = 32   # width of the path U-Nets (old UNet above is also base 32 now)

ARCHITECTURES = {
    # baseline: walk down one level at a time, walk back up
    "unet":              [0, 1, 2, 3, 2, 1, 0],
    # the idea, stepping: 1 down/1 up, 2 down/2 up, 3 down/3 up
    "ladder_step":       [0, 1, 0, 1, 2, 1, 0, 1, 2, 3, 2, 1, 0],
    # the idea, jumping: straight to depth k and straight back, k = 1, 2, 3
    "ladder_jump":       [0, 1, 0, 2, 0, 3, 0],
    # same jumps but coarse-to-fine: global layout first, details last
    "ladder_reverse":    [0, 3, 0, 2, 0, 1, 0],
    # jump down to depth k, walk back up one level at a time
    "jump_down_step_up": [0, 1, 0, 2, 1, 0, 3, 2, 1, 0],
    # two full U's back to back (W-Net)
    "w_net":             [0, 1, 2, 3, 2, 1, 0, 1, 2, 3, 2, 1, 0],
}

# Sanity check: build each one and count parameters
x_dummy = tf.zeros((2, 32, 32, 3))
t_dummy = tf.constant([0, timesteps - 1])
print(f"{'old UNet (base 32)':20s} {model.count_params():>11,}")
for name, levels in ARCHITECTURES.items():
    m = PathUNet(levels, base=PATH_BASE)
    assert m([x_dummy, t_dummy]).shape == (2, 32, 32, 3)
    print(f"{name:20s} {m.count_params():>11,}")
del m

## Train every path U-Net

Same data pipeline (`dataset` from the Train cell, random flips, batch 128), same optimizer and EMA
as the Gaussian `UNet`, and the same number of optimizer steps as its `epochs`.
Each variant checkpoints to Drive every `PATH_CKPT_EVERY` steps and is skipped once finished,
so re-running this cell after a disconnect just continues.

In [ ]:
PATH_CKPT_ROOT = os.path.join(os.path.dirname(CACHE_PATH), "checkpoints_path_unet")
os.makedirs(PATH_CKPT_ROOT, exist_ok=True)

PATH_STEPS = epochs * (len(x_train) // 128)   # same step count as the baseline UNet
PATH_CKPT_EVERY = 500

train_data = iter(dataset.repeat())


def build_path_model(name):
    """Model + optimizer + EMA + Drive checkpoint for one architecture."""
    net = PathUNet(ARCHITECTURES[name], base=PATH_BASE)
    net([tf.zeros((1, 32, 32, 3)), tf.zeros((1,), tf.int32)])            # build weights
    opt = tf.keras.optimizers.Adam(learning_rate=2e-4)
    opt.build(net.trainable_variables)                                 # build before tf.function / restore
    ema = [tf.Variable(v, trainable=False) for v in net.trainable_variables]
    ema_n = tf.Variable(0.0, trainable=False)
    step = tf.Variable(0, dtype=tf.int64, trainable=False)
    ckpt = tf.train.Checkpoint(model=net, optimizer=opt, ema_weights=ema, ema_step=ema_n, step=step)
    manager = tf.train.CheckpointManager(ckpt, os.path.join(PATH_CKPT_ROOT, name), max_to_keep=2)
    if manager.latest_checkpoint:
        ckpt.restore(manager.latest_checkpoint).expect_partial()
    return dict(model=net, optimizer=opt, ema=ema, ema_step=ema_n, step=step, manager=manager)


def make_path_train_step(p):
    net, opt, ema, ema_n = p["model"], p["optimizer"], p["ema"], p["ema_step"]

    @tf.function(autograph=False)
    def step_fn(x_0):
        b = tf.shape(x_0)[0]
        t = tf.random.uniform((b,), 0, timesteps, dtype=tf.int32)
        noise = tf.random.normal(tf.shape(x_0))
        a = tf.gather(ALPHA_BAR, t)[:, None, None, None]
        x_t = tf.sqrt(a) * x_0 + tf.sqrt(1.0 - a) * noise
        with tf.GradientTape() as tape:
            loss = loss_fn(noise, net([x_t, t], training=True))
        grads = tape.gradient(loss, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        ema_n.assign_add(1.0)
        d = tf.minimum(ema_decay, (1.0 + ema_n) / (10.0 + ema_n))
        for e, v in zip(ema, net.trainable_variables):
            e.assign(d * e + (1.0 - d) * v)
        return loss
    return step_fn


path_train_info = {}
for name in ARCHITECTURES:
    p = build_path_model(name)
    start = int(p["step"].numpy())
    if start >= PATH_STEPS:
        print(f"{name:20s} already trained ({start} steps), skipping")
        continue

    print(f"{name:20s} training steps {start} -> {PATH_STEPS}")
    step_fn = make_path_train_step(p)
    losses, t0 = [], time.time()
    for s in range(start, PATH_STEPS):
        losses.append(float(step_fn(next(train_data))))
        p["step"].assign(s + 1)
        if (s + 1) % PATH_CKPT_EVERY == 0 or s + 1 == PATH_STEPS:
            p["manager"].save()
            print(f"   step {s + 1:6d}  loss {np.mean(losses[-PATH_CKPT_EVERY:]):.4f}")
    path_train_info[name] = (time.time() - t0) / max(1, PATH_STEPS - start)
    print(f"   {path_train_info[name] * 1000:.1f} ms/step")

    del p, step_fn
    gc.collect()

## FID and Inception Score

- **FID** (lower is better): distance between Inception-V3 feature statistics of the generated images and of the
  real airplane training images. Measures both quality and diversity.
- **IS** (higher is better): how confidently and how diversely Inception-V3 classifies the samples.
  It was designed for all 1000 ImageNet classes, so on a single class (airplanes) it mostly rewards sharpness;
  treat FID as the main number.

Every model gets the **same `N_EVAL` samples, the same starting noise (fixed seed) and the same deterministic DDIM sampler**,
so the differences come from the network only. FID is biased upward at small sample counts, so only compare
numbers computed with the same `N_EVAL`. Uses Keras' InceptionV3, so absolute values differ slightly from
published ones; the real-data reference row shows what a perfect model would score.

In [ ]:
N_EVAL = 5000          # samples per model; same for every model
EVAL_DDIM_STEPS = 100
EVAL_BATCH = 500
EVAL_SEED = 1234

FID_REF_PATH = os.path.join(os.path.dirname(CACHE_PATH), "fid_reference_airplanes.npz")
EVAL_RESULTS_PATH = os.path.join(PATH_CKPT_ROOT, f"eval_results_n{N_EVAL}_ddim{EVAL_DDIM_STEPS}.json")


class InceptionScorer:
    """Pool features (2048-d) for FID and class probabilities for IS."""

    def __init__(self):
        base = tf.keras.applications.InceptionV3(include_top=True, weights="imagenet")
        self.model = tf.keras.Model(base.input, [base.get_layer("avg_pool").output, base.output])

    def __call__(self, images_01, batch=50):
        """images_01: float array in [0, 1], shape (N, 32, 32, 3)."""
        feats, probs = [], []
        for i in range(0, len(images_01), batch):
            x = tf.convert_to_tensor(images_01[i:i + batch], tf.float32)
            x = tf.image.resize(x, (299, 299), method="bilinear") * 2.0 - 1.0
            f, p = self.model(x, training=False)
            feats.append(f.numpy())
            probs.append(p.numpy())
        return np.concatenate(feats).astype(np.float64), np.concatenate(probs).astype(np.float64)


def fid_from_features(feats, mu_ref, sigma_ref):
    mu = feats.mean(axis=0)
    sigma = np.cov(feats, rowvar=False)
    diff = mu - mu_ref
    covmean = scipy.linalg.sqrtm(sigma.dot(sigma_ref))
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma.shape[0]) * 1e-6
        covmean = scipy.linalg.sqrtm((sigma + offset).dot(sigma_ref + offset))
    covmean = np.real(covmean)
    return float(diff.dot(diff) + np.trace(sigma) + np.trace(sigma_ref) - 2.0 * np.trace(covmean))


def inception_score(probs, splits=10):
    scores = []
    for part in np.array_split(probs, min(splits, len(probs))):
        py = part.mean(axis=0, keepdims=True)
        kl = (part * (np.log(part + 1e-12) - np.log(py + 1e-12))).sum(axis=1).mean()
        scores.append(np.exp(kl))
    return float(np.mean(scores)), float(np.std(scores))


scorer = InceptionScorer()

# Real-data reference: all training airplanes, cached on Drive next to the dataset
if os.path.exists(FID_REF_PATH):
    ref = np.load(FID_REF_PATH)
    mu_ref, sigma_ref = ref["mu"], ref["sigma"]
    real_is = (float(ref["is_mean"]), float(ref["is_std"]))
else:
    real_feats, real_probs = scorer(to_display(x_train))
    mu_ref, sigma_ref = real_feats.mean(axis=0), np.cov(real_feats, rowvar=False)
    real_is = inception_score(real_probs)
    np.savez(FID_REF_PATH, mu=mu_ref, sigma=sigma_ref, is_mean=real_is[0], is_std=real_is[1])
print(f"Real airplanes: IS {real_is[0]:.2f} +- {real_is[1]:.2f}  ({len(x_train)} images)")


def ddim_generate(predict_fn, n_images, n_steps=EVAL_DDIM_STEPS, batch=EVAL_BATCH, seed=EVAL_SEED):
    """Deterministic DDIM (eta=0), same maths as ddim_sample above, for any model.

    predict_fn(x_t, t) takes integer timesteps of shape (B,). The starting noise
    depends only on `seed`, so every model starts from exactly the same noise.
    """
    rng = np.random.default_rng(seed)
    step_indices = np.linspace(0, timesteps - 1, n_steps).round().astype(int)[::-1]
    out = []
    for start in range(0, n_images, batch):
        n = min(batch, n_images - start)
        x_t = tf.constant(rng.standard_normal((n, 32, 32, 3)), tf.float32)
        for i, t in enumerate(step_indices):
            t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else -1
            a_t = ALPHA_BAR[t]
            a_prev = ALPHA_BAR[t_prev] if t_prev >= 0 else tf.constant(1.0)
            eps = predict_fn(x_t, tf.fill([n], int(t)))
            x0 = tf.clip_by_value((x_t - tf.sqrt(1.0 - a_t) * eps) / tf.sqrt(a_t), -1.0, 1.0)
            x_t = tf.sqrt(a_prev) * x0 + tf.sqrt(1.0 - a_prev) * eps
        out.append(to_display(x_t.numpy()))
    return np.concatenate(out)


def evaluate(predict_fn):
    t0 = time.time()
    images = ddim_generate(predict_fn, N_EVAL)
    feats, probs = scorer(images)
    is_mean, is_std = inception_score(probs)
    return dict(fid=fid_from_features(feats, mu_ref, sigma_ref), is_mean=is_mean, is_std=is_std,
                n_samples=N_EVAL, seconds=time.time() - t0), images[:16]

In [ ]:
eval_results = json.load(open(EVAL_RESULTS_PATH)) if os.path.exists(EVAL_RESULTS_PATH) else {}
eval_previews = {}

# 1) Baseline: the Gaussian UNet trained above (EMA weights), same sampler and noise
if "UNet (baseline above)" not in eval_results:
    backup = swap_in_ema()
    baseline_predict = tf.function(lambda x, t: model([x, tf.gather(PE, t)], training=False))
    eval_results["UNet (baseline above)"], eval_previews["UNet (baseline above)"] = evaluate(baseline_predict)
    eval_results["UNet (baseline above)"]["params"] = model.count_params()
    restore_weights(backup)
    json.dump(eval_results, open(EVAL_RESULTS_PATH, "w"), indent=2)

# 2) Every path U-Net (EMA weights)
for name in ARCHITECTURES:
    if name in eval_results:
        continue
    p = build_path_model(name)
    if int(p["step"].numpy()) < PATH_STEPS:
        print(f"{name}: not fully trained yet ({int(p['step'].numpy())}/{PATH_STEPS} steps), skipping")
        continue
    for v, e in zip(p["model"].trainable_variables, p["ema"]):
        v.assign(e)
    net = p["model"]
    predict_fn = tf.function(lambda x, t: net([x, t], training=False))
    eval_results[name], eval_previews[name] = evaluate(predict_fn)
    eval_results[name]["params"] = net.count_params()
    print(f"{name:22s} FID {eval_results[name]['fid']:7.2f}   IS {eval_results[name]['is_mean']:.2f}")
    json.dump(eval_results, open(EVAL_RESULTS_PATH, "w"), indent=2)
    del p, net, predict_fn
    gc.collect()

print(f"\nAll models: {N_EVAL} samples, DDIM {EVAL_DDIM_STEPS} steps, seed {EVAL_SEED}")
print(f"{'model':22s} {'params':>10s} {'FID (lower)':>12s} {'IS (higher)':>16s}")
print(f"{'real airplanes':22s} {'-':>10s} {'0.00':>12s} {real_is[0]:9.2f} +- {real_is[1]:.2f}")
for name, r in sorted(eval_results.items(), key=lambda kv: kv[1]["fid"]):
    print(f"{name:22s} {r['params']:>10,} {r['fid']:12.2f} {r['is_mean']:9.2f} +- {r['is_std']:.2f}")

In [ ]:
names = sorted(eval_results, key=lambda k: eval_results[k]["fid"])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(names, [eval_results[n]["fid"] for n in names])
ax[0].set_title(f"FID, lower is better ({N_EVAL} samples each)")
ax[0].invert_yaxis()
ax[1].barh(names, [eval_results[n]["is_mean"] for n in names],
           xerr=[eval_results[n]["is_std"] for n in names])
ax[1].axvline(real_is[0], color="gray", linestyle="--", label="real airplanes")
ax[1].set_title("Inception Score, higher is better")
ax[1].invert_yaxis()
ax[1].legend()
plt.tight_layout()
plt.show()

# Same starting noise for every model, so each column is the "same" sample
for n in names:
    if n in eval_previews:
        show_images(eval_previews[n][:8], title=f"{n}: FID {eval_results[n]['fid']:.1f}", cols=8)

## Size-matched comparisons

- `ladder_step_b24`: same parameters as `unet` (1.54M). Does the stepping shape win without extra size?
- `unet_b40`: about the same parameters as `ladder_step`. Can a bigger plain U-Net close the gap?

Trained exactly like the path U-Nets above (same `train_data`, same step count), saved under `checkpoints_path_unet/`.

In [ ]:
# Size-matched comparisons:
#   ladder_step_b24 -> same params as unet (base 32): does the shape win without extra size?
#   unet_b40        -> about the same params as ladder_step (base 32): can a bigger plain U-Net close the gap?
EXTRA_ARCHITECTURES = {
    "ladder_step_b24": ([0, 1, 0, 1, 2, 1, 0, 1, 2, 3, 2, 1, 0], 24),
    "unet_b40":        ([0, 1, 2, 3, 2, 1, 0], 40),
}


def build_extra_model(name):
    """Same as build_path_model, but each architecture has its own width."""
    levels, base = EXTRA_ARCHITECTURES[name]
    net = PathUNet(levels, base=base)
    net([tf.zeros((1, 32, 32, 3)), tf.zeros((1,), tf.int32)])
    opt = tf.keras.optimizers.Adam(learning_rate=2e-4)
    opt.build(net.trainable_variables)
    ema = [tf.Variable(v, trainable=False) for v in net.trainable_variables]
    ema_n = tf.Variable(0.0, trainable=False)
    step = tf.Variable(0, dtype=tf.int64, trainable=False)
    ckpt = tf.train.Checkpoint(model=net, optimizer=opt, ema_weights=ema, ema_step=ema_n, step=step)
    manager = tf.train.CheckpointManager(ckpt, os.path.join(PATH_CKPT_ROOT, name), max_to_keep=2)
    if manager.latest_checkpoint:
        ckpt.restore(manager.latest_checkpoint).expect_partial()
    return dict(model=net, optimizer=opt, ema=ema, ema_step=ema_n, step=step, manager=manager)


print(f"{'unet (base 32)':20s} {'1,544,675':>11s}   <- reference")
print(f"{'ladder_step (base 32)':20s} {'2,353,059':>11s}   <- reference")
for name in EXTRA_ARCHITECTURES:
    p = build_extra_model(name)
    print(f"{name:20s} {p['model'].count_params():>11,}")
    del p
gc.collect()

In [ ]:
for name in EXTRA_ARCHITECTURES:
    p = build_extra_model(name)
    start = int(p["step"].numpy())
    if start >= PATH_STEPS:
        print(f"{name:20s} already trained ({start} steps), skipping")
        continue

    print(f"{name:20s} training steps {start} -> {PATH_STEPS}")
    step_fn = make_path_train_step(p)
    losses, t0 = [], time.time()
    for s in range(start, PATH_STEPS):
        losses.append(float(step_fn(next(train_data))))
        p["step"].assign(s + 1)
        if (s + 1) % PATH_CKPT_EVERY == 0 or s + 1 == PATH_STEPS:
            p["manager"].save()
            print(f"   step {s + 1:6d}  loss {np.mean(losses[-PATH_CKPT_EVERY:]):.4f}")
    print(f"   {(time.time() - t0) / max(1, PATH_STEPS - start) * 1000:.1f} ms/step")

    del p, step_fn
    gc.collect()

In [ ]:
for name in EXTRA_ARCHITECTURES:
    if name in eval_results:
        continue
    p = build_extra_model(name)
    if int(p["step"].numpy()) < PATH_STEPS:
        print(f"{name}: not fully trained yet, skipping")
        continue
    for v, e in zip(p["model"].trainable_variables, p["ema"]):
        v.assign(e)
    net = p["model"]
    predict_fn = tf.function(lambda x, t: net([x, t], training=False))
    eval_results[name], eval_previews[name] = evaluate(predict_fn)
    eval_results[name]["params"] = net.count_params()
    print(f"{name:22s} FID {eval_results[name]['fid']:7.2f}   IS {eval_results[name]['is_mean']:.2f}")
    json.dump(eval_results, open(EVAL_RESULTS_PATH, "w"), indent=2)
    del p, net, predict_fn
    gc.collect()

print(f"\nAll models: {N_EVAL} samples, DDIM {EVAL_DDIM_STEPS} steps, seed {EVAL_SEED}")
print(f"{'model':22s} {'params':>10s} {'FID (lower)':>12s} {'IS (higher)':>16s}")
print(f"{'real airplanes':22s} {'-':>10s} {'0.00':>12s} {real_is[0]:9.2f} +- {real_is[1]:.2f}")
for name, r in sorted(eval_results.items(), key=lambda kv: kv[1]["fid"]):
    print(f"{name:22s} {r['params']:>10,} {r['fid']:12.2f} {r['is_mean']:9.2f} +- {r['is_std']:.2f}")

## Fixed validation loss by noise level

Every trained model scored on the same held-out test airplanes, with identical timesteps and noise.
The loss is split by noise level: low t = fine detail, high t = global layout. Trains nothing.

In [ ]:
# Fixed validation: held-out test airplanes, identical t and noise for every model,
# using EMA weights. Split by noise level to see WHERE models differ.
VAL_REPEATS = 4          # each test image used with 4 different (t, noise) pairs
BUCKETS = [(0, 100), (100, 300), (300, 600), (600, 1000)]   # low t = fine detail, high t = global layout

rng = np.random.default_rng(0)
xv = np.repeat(x_test.astype("float32") / 127.5 - 1.0, VAL_REPEATS, axis=0)
tv = rng.integers(0, timesteps, len(xv)).astype("int32")
nv = rng.standard_normal(xv.shape).astype("float32")
av = tf.gather(ALPHA_BAR, tv).numpy()[:, None, None, None]
xtv = (np.sqrt(av) * xv + np.sqrt(1.0 - av) * nv).astype("float32")


def fixed_val_loss(predict_fn, batch=500):
    errs = np.concatenate([
        tf.reduce_mean((predict_fn(xtv[i:i + batch], tv[i:i + batch]) - nv[i:i + batch]) ** 2,
                       axis=[1, 2, 3]).numpy()
        for i in range(0, len(xv), batch)
    ])
    out = {"all": errs.mean()}
    for lo, hi in BUCKETS:
        out[f"t{lo}-{hi}"] = errs[(tv >= lo) & (tv < hi)].mean()
    return out


val_table = {}

backup = swap_in_ema()
val_table["UNet (baseline above)"] = fixed_val_loss(
    tf.function(lambda x, t: model([x, tf.gather(PE, t)], training=False)))
restore_weights(backup)

builders = [(n, build_path_model) for n in ARCHITECTURES]
if "EXTRA_ARCHITECTURES" in globals():
    builders += [(n, build_extra_model) for n in EXTRA_ARCHITECTURES]

for name, build in builders:
    p = build(name)
    if int(p["step"].numpy()) == 0:
        continue                                   # not trained yet
    for v, e in zip(p["model"].trainable_variables, p["ema"]):
        v.assign(e)
    net = p["model"]
    val_table[name] = fixed_val_loss(tf.function(lambda x, t: net([x, t], training=False)))
    val_table[name]["steps"] = int(p["step"].numpy())
    del p, net
    gc.collect()

ref = val_table.get("unet", next(iter(val_table.values())))
cols = ["all"] + [f"t{lo}-{hi}" for lo, hi in BUCKETS]
print(f"{len(xv)} fixed samples; numbers in brackets = % vs path 'unet' (negative = better)\n")
print(f"{'model':22s}" + "".join(f"{c:>18s}" for c in cols))
for name, r in sorted(val_table.items(), key=lambda kv: kv[1]["all"]):
    print(f"{name:22s}" + "".join(f"{r[c]:9.4f} ({100 * (r[c] / ref[c] - 1):+5.1f}%)" for c in cols))

## Blur + noise diffusion for every architecture

Same architectures as above, trained on the blur + noise degradation (x0-prediction) for 150 epochs.
Every model sees the **identical** training stream (same images, flips, t and noise at each step), and the
original `UNet` is retrained the same way as the baseline. Saved under `checkpoints_path_unet_blur/`;
the existing `model_blur` and its checkpoints are not touched.

In [ ]:
# Blur + noise versions of every architecture, all trained on the IDENTICAL stream:
# training step s uses the same images, flips, t and noise for every model.
BLUR_ARCH_CKPT_ROOT = os.path.join(os.path.dirname(CACHE_PATH), "checkpoints_path_unet_blur")
os.makedirs(BLUR_ARCH_CKPT_ROOT, exist_ok=True)

BLUR_EPOCHS = 150                                    # was 100
BLUR_BATCH = 128
BLUR_STEPS_PER_EPOCH = len(x_train) // BLUR_BATCH    # 39
BLUR_STEPS = BLUR_EPOCHS * BLUR_STEPS_PER_EPOCH      # 5850
BLUR_CKPT_EVERY = 500

# name: (levels, base). "original" = the UNet class from the top of the notebook.
BLUR_ARCHITECTURES = {"original_unet": ("original", 32)}
BLUR_ARCHITECTURES.update({name: (levels, PATH_BASE) for name, levels in ARCHITECTURES.items()})

x_train_f32 = np.asarray(x_train, dtype=np.float32)


def blur_batch(step, seed=0):
    """Everything about training step `step` is fixed by (seed, step) alone."""
    epoch, i = divmod(step, BLUR_STEPS_PER_EPOCH)
    perm = np.random.default_rng([seed, 0, epoch]).permutation(len(x_train_f32))
    x = x_train_f32[perm[i * BLUR_BATCH:(i + 1) * BLUR_BATCH]]
    rng = np.random.default_rng([seed, 1, step])
    flip = rng.random(BLUR_BATCH) < 0.5
    x[flip] = x[flip][:, :, ::-1, :]
    t = rng.integers(0, timesteps, BLUR_BATCH).astype("int32")
    noise = rng.standard_normal(x.shape, dtype=np.float32)
    return x, t, noise


def build_blur_arch(name):
    """Model + optimizer + EMA + Drive checkpoint. p["forward"](x, t) takes integer t for every model."""
    levels, base = BLUR_ARCHITECTURES[name]
    tf.keras.utils.set_random_seed(0)
    if levels == "original":
        net = UNet(input_shape=(32, 32, 3), time_dim=embedding_dim, base=base)
        forward = lambda x, t, training=False: net([x, tf.gather(PE, t)], training=training)
    else:
        net = PathUNet(levels, base=base)
        forward = lambda x, t, training=False: net([x, t], training=training)
    forward(tf.zeros((1, 32, 32, 3)), tf.zeros((1,), tf.int32))         # build weights
    opt = tf.keras.optimizers.Adam(learning_rate=2e-4)
    opt.build(net.trainable_variables)
    ema = [tf.Variable(v, trainable=False) for v in net.trainable_variables]
    ema_n = tf.Variable(0.0, trainable=False)
    step = tf.Variable(0, dtype=tf.int64, trainable=False)
    ckpt = tf.train.Checkpoint(model=net, optimizer=opt, ema_weights=ema, ema_step=ema_n, step=step)
    manager = tf.train.CheckpointManager(ckpt, os.path.join(BLUR_ARCH_CKPT_ROOT, name), max_to_keep=2)
    if manager.latest_checkpoint:
        ckpt.restore(manager.latest_checkpoint).expect_partial()
    return dict(model=net, forward=forward, optimizer=opt, ema=ema, ema_step=ema_n, step=step, manager=manager)


def make_blur_arch_train_step(p):
    net, forward, opt, ema, ema_n = p["model"], p["forward"], p["optimizer"], p["ema"], p["ema_step"]

    @tf.function(autograph=False)
    def step_fn(x_0, t, noise):
        x_t = degrade_blur_noise(x_0, t, noise)
        with tf.GradientTape() as tape:
            loss = loss_fn(x_0, forward(x_t, t, training=True))      # x0-prediction, as in train_step_blur
        grads = tape.gradient(loss, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        ema_n.assign_add(1.0)
        d = tf.minimum(ema_decay, (1.0 + ema_n) / (10.0 + ema_n))
        for e, v in zip(ema, net.trainable_variables):
            e.assign(d * e + (1.0 - d) * v)
        return loss
    return step_fn


for name in BLUR_ARCHITECTURES:
    p = build_blur_arch(name)
    print(f"{name:20s} {p['model'].count_params():>11,} params")
    del p
gc.collect()

In [ ]:
for name in BLUR_ARCHITECTURES:
    p = build_blur_arch(name)
    start = int(p["step"].numpy())
    if start >= BLUR_STEPS:
        print(f"{name:20s} already trained ({start} steps), skipping")
        continue

    print(f"{name:20s} training steps {start} -> {BLUR_STEPS}")
    step_fn = make_blur_arch_train_step(p)
    losses, t0 = [], time.time()
    for s in range(start, BLUR_STEPS):
        losses.append(float(step_fn(*blur_batch(s))))
        p["step"].assign(s + 1)
        if (s + 1) % BLUR_CKPT_EVERY == 0 or s + 1 == BLUR_STEPS:
            p["manager"].save()
            print(f"   step {s + 1:6d}  loss {np.mean(losses[-BLUR_CKPT_EVERY:]):.4f}")
    print(f"   {(time.time() - t0) / len(losses) * 1000:.1f} ms/step")

    del p, step_fn
    gc.collect()

In [ ]:
# FID / IS for the blur models: same 5000 samples, same starting noise, same
# cold-diffusion sampler for every model (reuses scorer, mu_ref, sigma_ref from the FID cell).
BLUR_EVAL_PATH = os.path.join(BLUR_ARCH_CKPT_ROOT, f"eval_results_n{N_EVAL}_steps{EVAL_DDIM_STEPS}.json")


def cold_generate(predict_fn, n_images, n_steps=EVAL_DDIM_STEPS, batch=EVAL_BATCH, seed=EVAL_SEED):
    """Same maths as cold_sample above, for any model; starting noise depends only on `seed`."""
    rng = np.random.default_rng(seed)
    step_indices = np.linspace(0, timesteps - 1, n_steps).round().astype(int)[::-1]
    out = []
    for start in range(0, n_images, batch):
        n = min(batch, n_images - start)
        fixed_noise = tf.constant(rng.standard_normal((n, 32, 32, 3)), tf.float32)
        x_t = fixed_noise
        for i, t in enumerate(step_indices):
            t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else -1
            t_batch = tf.fill([n], int(t))
            x0_hat = tf.clip_by_value(predict_fn(x_t, t_batch), -1.0, 1.0)
            if t_prev >= 0:
                t_prev_batch = tf.fill([n], int(t_prev))
                x_t = (x_t - degrade_blur_noise(x0_hat, t_batch, fixed_noise)
                       + degrade_blur_noise(x0_hat, t_prev_batch, fixed_noise))
            else:
                x_t = x0_hat
        out.append(to_display(x_t.numpy()))
    return np.concatenate(out)


blur_eval_results = json.load(open(BLUR_EVAL_PATH)) if os.path.exists(BLUR_EVAL_PATH) else {}
blur_eval_previews = {}

for name in BLUR_ARCHITECTURES:
    if name in blur_eval_results:
        print(f"{name:20s} already evaluated, FID {blur_eval_results[name]['fid']:.2f}")
        continue
    p = build_blur_arch(name)
    if int(p["step"].numpy()) < BLUR_STEPS:
        print(f"{name:20s} not fully trained yet, skipping")
        continue
    print(f"{name:20s} evaluating...", flush=True)
    for v, e in zip(p["model"].trainable_variables, p["ema"]):
        v.assign(e)
    fwd = p["forward"]
    predict_fn = tf.function(lambda x, t: fwd(x, t))

    t0 = time.time()
    images = cold_generate(predict_fn, N_EVAL)
    feats, probs = scorer(images)
    is_mean, is_std = inception_score(probs)
    blur_eval_results[name] = dict(fid=fid_from_features(feats, mu_ref, sigma_ref), is_mean=is_mean,
                                   is_std=is_std, n_samples=N_EVAL, params=p["model"].count_params(),
                                   seconds=time.time() - t0)
    blur_eval_previews[name] = images[:16]
    print(f"{name:20s} FID {blur_eval_results[name]['fid']:7.2f}   IS {is_mean:.2f}   "
          f"({blur_eval_results[name]['seconds'] / 60:.1f} min)")
    json.dump(blur_eval_results, open(BLUR_EVAL_PATH, "w"), indent=2)
    del p, fwd, predict_fn, images
    gc.collect()

print(f"\nBlur + noise, all models: {N_EVAL} samples, {EVAL_DDIM_STEPS} sampling steps, seed {EVAL_SEED}")
print(f"{'model':20s} {'params':>10s} {'FID (lower)':>12s} {'IS (higher)':>16s}")
print(f"{'real airplanes':20s} {'-':>10s} {'0.00':>12s} {real_is[0]:9.2f} +- {real_is[1]:.2f}")
for name, r in sorted(blur_eval_results.items(), key=lambda kv: kv[1]["fid"]):
    print(f"{name:20s} {r['params']:>10,} {r['fid']:12.2f} {r['is_mean']:9.2f} +- {r['is_std']:.2f}")

for name in sorted(blur_eval_previews, key=lambda k: blur_eval_results[k]["fid"]):
    show_images(blur_eval_previews[name][:8], title=f"blur {name}: FID {blur_eval_results[name]['fid']:.1f}", cols=8)

## Noise-routed U-Net

Every level has two blocks (a, b) going down and two going up; **which blocks run depends on the noise level**:

| t | blocks used (down, mirrored on the way up) |
|---|---|
| 0-100 | L0a, L0b, L1a, L1b, bottleneck |
| 100-250 | L1a, L2a, L3a, bottleneck |
| 250-400 | L2a, L2b, L3a, bottleneck |
| 400-1000 | L2a, L3a, L3b, bottleneck |

Four models on the identical training stream: the original `UNet` (baseline), `routed_dense` (all blocks
always, quality upper bound), `routed_fixed` (the table above), `routed_learned` (each block has a learned
gate g(t); closed blocks are skipped at sampling). Saved under `checkpoints_routed_unet/`.

In [ ]:
# Noise-routed U-Net: every level has two blocks (a, b) going down and two going up,
# and WHICH blocks run depends on the noise level t.
#   level 0 = 32x32, 1 = 16x16, 2 = 8x8 (+attention), 3 = 4x4, then a 4x4 bottleneck (+attention).
# Levels with no blocks in a route are still passed through (resize + skip connection),
# so every route keeps a full-resolution path from input to output.

ROUTES = [  # (t_from, t_to, blocks used at each level, same letters going down and up)
    (0,   100,  {0: "ab", 1: "ab", 2: "",   3: ""}),     # low noise: fine detail
    (100, 250,  {0: "",   1: "a",  2: "a",  3: "a"}),
    (250, 400,  {0: "",   1: "",   2: "ab", 3: "a"}),
    (400, 1000, {0: "",   1: "",   2: "a",  3: "ab"}),    # high noise: global layout
]
DENSE_ROUTE = {0: "ab", 1: "ab", 2: "ab", 3: "ab"}

ROUTED_BLOCKS = [f"{side}{level}{k}" for side in "ed" for level in range(4) for k in "ab"]   # 16 blocks
# Relative cost of one block at each level (channels^2 * pixels), used for the compute penalty/report
ROUTED_LEVEL_COST = {0: 1.0, 1: 1.0, 2: 0.33, 3: 0.25}    # level 2 includes its attention


def route_for_t(t_value):
    for lo, hi, route in ROUTES:
        if lo <= t_value < hi:
            return route
    return ROUTES[-1][2]


def route_cost(route):
    """Fraction of the dense model's block compute that a route uses."""
    used = sum(2 * ROUTED_LEVEL_COST[l] * len(route[l]) for l in range(4))     # x2: down + up
    return used / sum(2 * ROUTED_LEVEL_COST[l] * 2 for l in range(4))


class RoutedUNet(tf.keras.Model):
    """mode = "dense"   : every block at every t (upper bound on quality)
              "fixed"   : hand-designed ROUTES by t
              "learned" : every block has a gate g(t) in [0, 1]; training is pushed to
                          close gates it doesn't need, and closed blocks are skipped at sampling.
    """
    def __init__(self, mode, base=32, mults=(1, 2, 2, 4), time_dim=128):
        super().__init__()
        self.mode = mode
        self.time_dim = time_dim
        ch = [base * m for m in mults]
        self.ch = ch

        self.time_mlp = tf.keras.Sequential([
            layers.Dense(time_dim * 4, activation="swish"),
            layers.Dense(time_dim * 4),
        ])
        self.stem = layers.Conv2D(ch[0], 3, padding="same")
        self.downs = [layers.Conv2D(ch[l], 3, strides=2, padding="same") for l in (1, 2, 3)]
        self.ups = [tf.keras.Sequential([layers.UpSampling2D(2, interpolation="nearest"),
                                         layers.Conv2D(ch[l - 1], 3, padding="same")]) for l in (1, 2, 3)]
        self.fuse = [layers.Conv2D(ch[l], 1) for l in range(4)]        # merges skip + upsampled, every route

        self.blocks = {name: PResBlock(ch[int(name[1])], ch[int(name[1])]) for name in ROUTED_BLOCKS}
        self.attns = {name: PAttentionBlock(ch[2]) for name in ROUTED_BLOCKS if name[1] == "2"}

        self.mid1 = PResBlock(ch[3], ch[3])
        self.mid_attn = PAttentionBlock(ch[3])
        self.mid2 = PResBlock(ch[3], ch[3])

        if mode == "learned":
            # starts with every gate ~0.88 (nearly dense); training decides what to close
            self.gate_head = layers.Dense(len(ROUTED_BLOCKS), kernel_initializer="zeros",
                                          bias_initializer=tf.keras.initializers.Constant(2.0))

        self.out_norm = layers.GroupNormalization(groups=8)
        self.out_conv = layers.Conv2D(3, 3, padding="same")

    def gate_values(self, t):
        """(B, 16) gate values for integer timesteps t (learned mode only)."""
        return tf.sigmoid(self.gate_head(self.time_mlp(timestep_embedding(t, self.time_dim))))

    def forward(self, x, t, route=None, active=None, training=False, return_gates=False):
        """route: which blocks run (fixed mode). active: set of block names to run (pruned learned mode)."""
        route = route or DENSE_ROUTE
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim), training=training)
        gates = tf.sigmoid(self.gate_head(t_emb)) if self.mode == "learned" else None

        def run(name, h):
            if name[2] not in route[int(name[1])] or (active is not None and name not in active):
                return h
            y = self.blocks[name](h, t_emb)
            if name in self.attns:
                y = self.attns[name](y)
            if gates is not None:
                g = gates[:, ROUTED_BLOCKS.index(name)][:, None, None, None]
                y = h + g * (y - h)
            return y

        h = self.stem(x)
        skips = []
        for level in range(4):
            if level > 0:
                h = self.downs[level - 1](h)
            for k in "ab":
                h = run(f"e{level}{k}", h)
            skips.append(h)

        h = self.mid2(self.mid_attn(self.mid1(h, t_emb)), t_emb)

        for level in (3, 2, 1, 0):
            h = self.fuse[level](tf.concat([h, skips[level]], axis=-1))
            for k in "ab":
                h = run(f"d{level}{k}", h)
            if level > 0:
                h = self.ups[level - 1](h)

        out = self.out_conv(tf.nn.silu(self.out_norm(h)))
        return (out, gates) if return_gates else out

    def call(self, inputs, training=False):
        x, t = inputs
        return self.forward(x, t, training=training)


for mode in ("dense", "fixed", "learned"):
    m = RoutedUNet(mode, base=PATH_BASE)
    m([tf.zeros((2, 32, 32, 3)), tf.constant([0, 999])])
    print(f"routed_{mode:8s} {m.count_params():>11,} params")
for lo, hi, route in ROUTES:
    print(f"   t {lo:4d}-{hi:4d}: uses {route_cost(route) * 100:4.0f}% of the dense block compute")
del m

In [ ]:
# Train: original UNet, dense, fixed routing, learned routing - all on the IDENTICAL stream
# (training step s uses the same images, flips, t and noise for every model).
ROUTED_CKPT_ROOT = os.path.join(os.path.dirname(CACHE_PATH), "checkpoints_routed_unet")
os.makedirs(ROUTED_CKPT_ROOT, exist_ok=True)

ROUTED_EPOCHS = 100                                  # same as the earlier Gaussian runs
ROUTED_BATCH = 128
ROUTED_STEPS_PER_EPOCH = len(x_train) // ROUTED_BATCH
ROUTED_STEPS = ROUTED_EPOCHS * ROUTED_STEPS_PER_EPOCH  # 3900
ROUTED_CKPT_EVERY = 500
GATE_PENALTY = 0.01      # learned mode: how hard to push gates closed (raise if gates stay ~1, lower if all close)

ROUTED_MODELS = ["original_unet", "routed_dense", "routed_fixed", "routed_learned"]
x_train_f32 = np.asarray(x_train, dtype=np.float32)


def stream_batch(step, seed=0):
    """Everything about training step `step` is fixed by (seed, step) alone."""
    epoch, i = divmod(step, ROUTED_STEPS_PER_EPOCH)
    perm = np.random.default_rng([seed, 0, epoch]).permutation(len(x_train_f32))
    x = x_train_f32[perm[i * ROUTED_BATCH:(i + 1) * ROUTED_BATCH]]
    rng = np.random.default_rng([seed, 1, step])
    flip = rng.random(ROUTED_BATCH) < 0.5
    x[flip] = x[flip][:, :, ::-1, :]
    t = rng.integers(0, timesteps, ROUTED_BATCH).astype("int32")
    noise = rng.standard_normal(x.shape, dtype=np.float32)
    return x, t, noise


def build_routed(name):
    tf.keras.utils.set_random_seed(0)
    if name == "original_unet":
        net = UNet(input_shape=(32, 32, 3), time_dim=embedding_dim, base=32)
        net([tf.zeros((1, 32, 32, 3)), tf.zeros((1, embedding_dim))])
    else:
        net = RoutedUNet(name.split("_")[1], base=PATH_BASE)
        net([tf.zeros((1, 32, 32, 3)), tf.zeros((1,), tf.int32)])        # builds every block
    opt = tf.keras.optimizers.Adam(learning_rate=2e-4)
    opt.build(net.trainable_variables)
    ema = [tf.Variable(v, trainable=False) for v in net.trainable_variables]
    ema_n = tf.Variable(0.0, trainable=False)
    step = tf.Variable(0, dtype=tf.int64, trainable=False)
    ckpt = tf.train.Checkpoint(model=net, optimizer=opt, ema_weights=ema, ema_step=ema_n, step=step)
    manager = tf.train.CheckpointManager(ckpt, os.path.join(ROUTED_CKPT_ROOT, name), max_to_keep=2)
    if manager.latest_checkpoint:
        ckpt.restore(manager.latest_checkpoint).expect_partial()
    return dict(name=name, model=net, optimizer=opt, ema=ema, ema_step=ema_n, step=step, manager=manager)


def routed_loss(name, net, x_t, t, noise, training):
    """Noise-prediction MSE (+ gate penalty for learned routing). Returns (total, mse)."""
    if name == "original_unet":
        mse = loss_fn(noise, net([x_t, tf.gather(PE, t)], training=training))
        return mse, mse
    if name == "routed_fixed":
        # each sample goes through the route of its own t: split the batch by route
        sq = 0.0
        for lo, hi, route in ROUTES:
            m = (t >= lo) & (t < hi)
            pred = net.forward(tf.boolean_mask(x_t, m), tf.boolean_mask(t, m), route=route, training=training)
            sq += tf.reduce_sum(tf.square(pred - tf.boolean_mask(noise, m)))
        mse = sq / tf.cast(tf.size(noise), tf.float32)
        return mse, mse
    pred, gates = net.forward(x_t, t, training=training, return_gates=True)
    mse = loss_fn(noise, pred)
    if gates is None:
        return mse, mse
    cost = tf.constant([2 * ROUTED_LEVEL_COST[int(b[1])] for b in ROUTED_BLOCKS], tf.float32)
    used = tf.reduce_mean(tf.reduce_sum(gates * cost, axis=1)) / tf.reduce_sum(cost)
    return mse + GATE_PENALTY * used, mse


def make_routed_step(p):
    name, net, opt, ema, ema_n = p["name"], p["model"], p["optimizer"], p["ema"], p["ema_step"]

    @tf.function(autograph=False)
    def step_fn(x_0, t, noise):
        a = tf.gather(ALPHA_BAR, t)[:, None, None, None]
        x_t = tf.sqrt(a) * x_0 + tf.sqrt(1.0 - a) * noise
        with tf.GradientTape() as tape:
            total, mse = routed_loss(name, net, x_t, t, noise, training=True)
        grads = tape.gradient(total, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        ema_n.assign_add(1.0)
        d = tf.minimum(ema_decay, (1.0 + ema_n) / (10.0 + ema_n))
        for e, v in zip(ema, net.trainable_variables):
            e.assign(d * e + (1.0 - d) * v)
        return mse
    return step_fn


for name in ROUTED_MODELS:
    p = build_routed(name)
    start = int(p["step"].numpy())
    if start >= ROUTED_STEPS:
        print(f"{name:16s} already trained ({start} steps), skipping")
        continue
    print(f"{name:16s} {p['model'].count_params():>10,} params   training steps {start} -> {ROUTED_STEPS}")
    step_fn = make_routed_step(p)
    losses, t0 = [], time.time()
    for s in range(start, ROUTED_STEPS):
        losses.append(float(step_fn(*stream_batch(s))))
        p["step"].assign(s + 1)
        if (s + 1) % ROUTED_CKPT_EVERY == 0 or s + 1 == ROUTED_STEPS:
            p["manager"].save()
            print(f"   step {s + 1:6d}  loss {np.mean(losses[-ROUTED_CKPT_EVERY:]):.4f}")
    print(f"   {(time.time() - t0) / len(losses) * 1000:.1f} ms/step")
    del p, step_fn
    gc.collect()

In [ ]:
# FID / IS (same 5000 samples, same starting noise, same DDIM sampler for every model),
# plus how much block compute each model spends per sampling step.
ROUTED_EVAL_PATH = os.path.join(ROUTED_CKPT_ROOT, f"eval_results_n{N_EVAL}_ddim{EVAL_DDIM_STEPS}.json")
PRUNE_BELOW = 0.1        # learned routing: skip a block at timestep t if its gate < this
sample_ts = np.linspace(0, timesteps - 1, EVAL_DDIM_STEPS).round().astype(int)
block_cost = np.array([2 * ROUTED_LEVEL_COST[int(b[1])] for b in ROUTED_BLOCKS])


def routed_predictors(p):
    """Yields (label, predict_fn, fraction of dense block compute used during sampling)."""
    name, net = p["name"], p["model"]
    if name == "original_unet":
        yield name, tf.function(lambda x, t: net([x, tf.gather(PE, t)], training=False)), None
    elif name == "routed_dense":
        yield name, tf.function(lambda x, t: net.forward(x, t)), 1.0
    elif name == "routed_fixed":
        per_route = [tf.function(lambda x, t, r=route: net.forward(x, t, route=r)) for _, _, route in ROUTES]
        def predict(x, t):
            tv = int(t[0])
            return per_route[next(i for i, (lo, hi, _) in enumerate(ROUTES) if lo <= tv < hi or i == len(ROUTES) - 1)](x, t)
        yield name, predict, float(np.mean([route_cost(route_for_t(tv)) for tv in sample_ts]))
    else:
        yield name + " (all blocks)", tf.function(lambda x, t: net.forward(x, t)), 1.0
        gates = net.gate_values(tf.constant(sample_ts, tf.int32)).numpy()        # (steps, 16)
        active_by_t = {int(tv): frozenset(b for b, g in zip(ROUTED_BLOCKS, row) if g >= PRUNE_BELOW)
                       for tv, row in zip(sample_ts, gates)}
        cache = {}
        def predict(x, t):
            active = active_by_t[int(t[0])]
            if active not in cache:
                cache[active] = tf.function(lambda x, t, a=active: net.forward(x, t, active=a))
            return cache[active](x, t)
        used = np.mean([block_cost[[b in active_by_t[int(tv)] for b in ROUTED_BLOCKS]].sum() / block_cost.sum()
                        for tv in sample_ts])
        yield name + " (pruned)", predict, float(used)


routed_results = json.load(open(ROUTED_EVAL_PATH)) if os.path.exists(ROUTED_EVAL_PATH) else {}
routed_previews = {}

for name in ROUTED_MODELS:
    p = build_routed(name)
    if int(p["step"].numpy()) < ROUTED_STEPS:
        print(f"{name:28s} not fully trained yet, skipping")
        continue
    for v, e in zip(p["model"].trainable_variables, p["ema"]):
        v.assign(e)

    if name == "routed_learned":      # where did it learn to spend compute?
        t_grid = np.arange(0, timesteps, 10)
        g = p["model"].gate_values(tf.constant(t_grid, tf.int32)).numpy().T
        plt.figure(figsize=(10, 4.5))
        plt.imshow(g, aspect="auto", cmap="viridis", vmin=0, vmax=1,
                   extent=[0, timesteps, len(ROUTED_BLOCKS) - 0.5, -0.5])
        plt.yticks(range(len(ROUTED_BLOCKS)), ROUTED_BLOCKS)
        plt.xlabel("timestep t (low noise -> high noise)")
        plt.title("learned gates per block (e = down, d = up; level 0 = 32x32 ... 3 = 4x4)")
        plt.colorbar(label="gate")
        plt.show()

    for label, predict_fn, used in routed_predictors(p):
        if label in routed_results:
            print(f"{label:28s} already evaluated, FID {routed_results[label]['fid']:.2f}")
            continue
        print(f"{label:28s} evaluating...", flush=True)
        r, previews = evaluate(predict_fn)
        r["params"] = p["model"].count_params()
        r["block_compute"] = used
        routed_results[label], routed_previews[label] = r, previews
        print(f"{label:28s} FID {r['fid']:7.2f}   IS {r['is_mean']:.2f}   sampling {r['seconds'] / 60:.1f} min")
        json.dump(routed_results, open(ROUTED_EVAL_PATH, "w"), indent=2)
    del p
    gc.collect()

print(f"\nAll models: {N_EVAL} samples, DDIM {EVAL_DDIM_STEPS} steps, seed {EVAL_SEED}")
print(f"{'model':28s} {'params':>10s} {'block compute':>14s} {'FID':>8s} {'IS':>14s} {'time':>8s}")
for label, r in sorted(routed_results.items(), key=lambda kv: kv[1]["fid"]):
    bc = "-" if r["block_compute"] is None else f"{r['block_compute'] * 100:.0f}%"
    print(f"{label:28s} {r['params']:>10,} {bc:>14s} {r['fid']:8.2f} {r['is_mean']:7.2f} +- {r['is_std']:.2f} "
          f"{r['seconds'] / 60:6.1f}m")

for label in sorted(routed_previews, key=lambda k: routed_results[k]["fid"]):
    show_images(routed_previews[label][:8], title=f"{label}: FID {routed_results[label]['fid']:.1f}", cols=8)